In [62]:
# Load env variables and create client
import os

from dotenv import load_dotenv
from anthropic import Anthropic

#Only when necessary
#load_dotenv(override=True)

load_dotenv()

client = Anthropic()
model = os.environ["ANTHROPIC_MODEL"]
print(model)
max_tokens = 1000

claude-sonnet-5-5


In [63]:
from pydantic import BaseModel

class FAQ(BaseModel):
    question: str
    answer: str
    citation: list[str]

In [ ]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": max_tokens,
        "messages": messages,
        "stop_sequences": stop_sequences,
        "output_format": list[FAQ],
        "thinking": { 
            "type": "between_tools" 
        },
        "output_config": { "effort": "low" }
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.parse(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

import json

def json_from_message(message):
    return json.loads("\n".join([block.text for block in message.content if block.type == "text"]))

In [93]:
# Prompt

prompt = """
Analyse the attached document that contains property rules and instructions following stepecif steps:

1. Question location: locate rules or instructions in the image by looking for:
 - Topics that may describe how something should work or be done
 - Texts that present a topic followed by an instructions or steps
 - That describes actions that the guest should follow or behave

2. Answer location: locate the answers to the previous questions by looking for:
 - Explanation of how things work or should be done
 - Description or manual of products, items, and things
 - They must be related to the question previous found

3. The format expected:
 - Question should be 50 character or less.
 - Answers should be 50 character or less.
 - You should be as consice as possible when extracting the texts and writing them as FAQ
 - Be straight to the point as our guest don't like long texts
 - You should return in JSON format containing three parameters: question, answer, citation
 - The citation should be the piece you read to extract the Question and the Answer. It must be the exact text, copied character by character.

4. Examples of common FAQ
  {
    "question": "What is the WiFi network name and password?",
    "answer": "The Wi-Fi details are listed underneath the router on the TV desk.",
    "citation": "Wi-Fi: the network name and password are on a sticker underneath the router on the TV desk."
  },
  {
    "question": "Are the shops open at the weekends here?",
    "answer": "Yes, most shops open on weekends, but hours may vary by store and location.",
    "citation": "Shopping: most shops in the area open on weekends, but hours vary by store."
  },
  {
    "question": "How much is street parking?",
    "answer": "Street parking rates vary by location; please check local signs for fees.",
    "citation": "Parking: street parking is paid; rates vary by street, so please check the local signs for fees."
  }

5. You must NEVER do the following:
 - Never create an FAQ that is not in the image
 - Never make up answers if you can't find one
 - Never make up questions if you can't find one

"""

In [78]:
# Process image
import base64

with open("../data/samples/houserules.png", "rb") as f:
    image_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

messages = []

add_user_message(messages, [
    # Image Block
    {
        "type": "image",
        "source": {
            "type": "base64",
            "media_type": "image/png",
            "data": image_bytes,
        }
    },
    # Text Block
    {
        "type": "text",
        "text": prompt
    }
])

response = chat(messages)

response

ParsedMessage[TypeVar](id='msg_011CgJ8NQD5GWnmZovEGoGYG', container=None, content=[ParsedTextBlock[TypeVar](citations=None, text='[{"question":"Is smoking or vaping allowed?","answer":"No smoking or vaping.","citation":["No smoking or vaping"]},{"question":"What are the quiet hours?","answer":"Be quiet after 10 pm to respect neighbors.","citation":["Please respect our neighbors, be quiet after 10 pm"]},{"question":"Are parties allowed?","answer":"No, strictly no parties allowed.","citation":["Strickly no parties allowed"]},{"question":"Are unregistered guests allowed?","answer":"No, unregistered guests are not allowed.","citation":["No unregistered guests allowed"]},{"question":"What should I do with trash before checkout?","answer":"Ensure all trash is removed before check out.","citation":["Please ensure all trash is removed before you check out"]},{"question":"What time is check out?","answer":"Check out time is at 11 am.","citation":["Check out time is at 11 am"]}]', type='text', p

In [84]:
text_from_message(response)

'[{"question":"Is smoking or vaping allowed?","answer":"No smoking or vaping.","citation":["No smoking or vaping"]},{"question":"What are the quiet hours?","answer":"Be quiet after 10 pm to respect neighbors.","citation":["Please respect our neighbors, be quiet after 10 pm"]},{"question":"Are parties allowed?","answer":"No, strictly no parties allowed.","citation":["Strickly no parties allowed"]},{"question":"Are unregistered guests allowed?","answer":"No, unregistered guests are not allowed.","citation":["No unregistered guests allowed"]},{"question":"What should I do with trash before checkout?","answer":"Ensure all trash is removed before check out.","citation":["Please ensure all trash is removed before you check out"]},{"question":"What time is check out?","answer":"Check out time is at 11 am.","citation":["Check out time is at 11 am"]}]'

In [94]:
# Process PDF
import base64

with open("../data/samples/houserules.pdf", "rb") as f:
    file_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

messages = []

add_user_message(
    messages,
    [
        {
            "type": "document",
            "source": {
                "type": "base64",
                "media_type": "application/pdf",
                "data": file_bytes,
            },
        },
        {
            "type": "text", 
            "text": prompt
        },
    ],
)

response = chat(messages)

response

ParsedMessage[TypeVar](id='msg_011CgJDiBcfWFRSEPLDdq9zk', container=None, content=[ParsedTextBlock[TypeVar](citations=None, text='[{"question":"Qual o WiFi e a senha?","answer":"Rede 701 / 701 2G. Senha: cerimonial","citation":["701 / 701 2G","Senha: cerimonial"]},{"question":"Como entrar no prédio?","answer":"Use o tag azul na 1ª porta; empurre a porta.","citation":["Posicione o tag azul para abrir a primeira porta de vidro, empurre a porta e automaticamente irá abrir a segunda porta."]},{"question":"Como abrir a porta do apartamento?","answer":"Não use a fechadura da parte superior.","citation":["Porta do apartamento: não usar a fechadura da parte superior."]},{"question":"Como funciona o interfone?","answer":"Interfonar 701; para liberar, digitar # 2.","citation":["A pessoa que desejam acessar ao apartamento deve interfonar: 701","Para liberar o acesso no interfone digitar: # 2"]},{"question":"Qual a regra sobre som e barulho?","answer":"Sem som alto/barulho após 22:00. Sujeito a mu

In [97]:
json_from_message(response)

AttributeError: 'str' object has no attribute 'read'